# 1. Avaliação independente do experimento
Selecione um experimento concluído. A leitura não cria treinamento e não altera seus artefatos.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from basket_price_prediction.config import ROOT, DATASET, SPLITS, FEATURES
from basket_price_prediction.data import load_data
from basket_price_prediction.preprocessing import prepare_monthly
from basket_price_prediction.features import build_supervised, temporal_split
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
import json
from IPython.display import Image
from basket_price_prediction.experiments import load_experiment, file_hash
from basket_price_prediction.evaluation import group_performance, calculate_metrics
from basket_price_prediction.config import RESULTS
completed=[p for p in RESULTS.iterdir() if p.is_dir() and (p/"manifest.json").exists() and json.loads((p/"manifest.json").read_text()).get("status")=="complete"]
if not completed: raise RuntimeError("Nenhum experimento concluído. Execute o treinamento no notebook 2.")
EXPERIMENT = sorted(completed)[-1]  # Substitua pelo caminho de uma execução para selecioná-la explicitamente.
exp=load_experiment(EXPERIMENT); test=exp["test"]; threshold=exp["config"]["selection"]["threshold"]
print("Experimento:",EXPERIMENT.name)
print("Família selecionada:",exp["config"]["selection"]["algorithm"],"| limiar:",threshold)

## 2. Verificação da fonte, separação e modelo salvo
O hash verifica o arquivo de origem. A igualdade das probabilidades reconstruídas verifica o pipeline e as mesmas observações do teste.

In [ ]:
assert file_hash(DATASET)==exp["manifest"]["dataset_sha256"], "A fonte mudou desde o treinamento"
monthly,_=prepare_monthly(load_data()); supervised,_=build_supervised(monthly); parts=temporal_split(supervised,SPLITS)
assert len(test)==len(parts["test"])
reloaded_probability=exp["model"].predict_proba(parts["test"][FEATURES])[:,1]
assert np.allclose(reloaded_probability,test.probability,rtol=1e-10,atol=1e-10)
assert test.target.tolist()==parts["test"].target.tolist()
print("Hash, registros, rótulos e probabilidades do modelo salvo conferem.")
display(pd.DataFrame(exp["summary"]["splits"]).T)

## 3. A comparação entre famílias
A coluna de validação é a que decidiu a seleção. A coluna de teste foi calculada **depois** que a escolha já estava fechada e não realimentou decisão alguma: ela existe por transparência. Trocar a família depois de ver o teste converteria o teste em conjunto de seleção e destruiria a única estimativa não enviesada disponível.

In [ ]:
comparacao=exp["comparison"].sort_values("f1_validation",ascending=False)
display(comparacao[["algorithm","params","threshold","f1_validation","precision_validation","recall_validation","f1_test","precision_test","recall_test"]])
display(Image(filename=str(EXPERIMENT/"figures"/"model_comparison.png")))
faixa=comparacao.f1_test.max()-comparacao.f1_test.min()
display(Markdown(f"Amplitude do F1 de teste entre as quatro famílias: **{faixa:.3f}**. Compare com a largura do intervalo de confiança do F1 na seção seguinte antes de afirmar que uma família é superior a outra."))

## 4. Métricas, referências e intervalos de confiança
Precisão = VP/(VP+FP); recall = VP/(VP+FN); F1 combina as duas. Acurácia pode favorecer a classe majoritária. As referências são regras descritivas no mesmo teste, não classificadores treinados.

Os intervalos usam **bootstrap por série**: cada reamostra sorteia séries inteiras com reposição, porque meses da mesma série não são observações independentes. Um intervalo calculado linha a linha seria estreito demais. A mesma reamostragem produz o intervalo da *diferença* entre o modelo e cada referência — é esse intervalo que diz se o ganho é real ou ruído amostral.

In [ ]:
planas=["test","validation_before_refit","reference_test_always_zero","reference_test_previous_increase"]
display(pd.DataFrame({k:exp["metrics"][k] for k in planas}).T)
print((EXPERIMENT/"reports"/"classification_report.txt").read_text())

In [ ]:
ic=exp["metrics"]["test_confidence_intervals"]
tabela=pd.DataFrame(ic["intervals"]).T[["low","median","high"]]
tabela.columns=[f"IC {(1-ic['level'])/2:.1%}","mediana",f"IC {(1+ic['level'])/2:.1%}"]
display(tabela.round(3))
display(Image(filename=str(EXPERIMENT/"figures"/"metric_intervals.png")))
for nome,rotulo in [("f1_gain_over_previous_increase","ganho de F1 sobre repetir a alta anterior"),("balanced_accuracy_gain_over_always_zero","ganho de acurácia balanceada sobre prever sempre sem alta")]:
    faixa=ic["intervals"][nome]
    veredito="exclui zero: ganho estatisticamente significativo" if faixa["low"]>0 else "contém zero: ganho não distinguível de ruído"
    display(Markdown(f"IC {ic['level']:.0%} do {rotulo}: **[{faixa['low']:+.3f}, {faixa['high']:+.3f}]** — {veredito}."))
print("Reamostragem de %d séries, %d reamostras válidas." % (ic["series"],ic["samples"]))

## 5. Matriz de confusão, curvas e probabilidades
A curva precisão–recall deve ser comparada com a prevalência. ROC-AUC mede ordenação; Brier avalia probabilidades. A calibração em decis é descritiva e não garante calibração futura.

In [ ]:
for name in ["confusion_matrix.png","precision_recall_curve.png","roc_curve.png","probability_distribution.png","calibration_curve.png","threshold_tradeoff.png"]:
    display(Image(filename=str(EXPERIMENT/"figures"/name)))

## 6. Desempenho temporal
Para cada mês-alvo são apresentados tamanho, positivos, precisão, recall, F1 e taxa prevista de positivos. Grupos sem positivos têm recall e F1 indefinidos. A ausência de data de coleta a partir de novembro/2025 limita a confirmação da disponibilidade real da informação na maior parte do teste.

In [ ]:
months=group_performance(test,"target_month",threshold).sort_values("target_month")
display(months[["target_month","n","positives","precision","recall","f1","positive_prediction_rate"]]); display(Image(filename=str(EXPERIMENT/"figures"/"monthly_performance.png")))
observed_figure = EXPERIMENT / "figures" / "monthly_observed_increases.png"
if observed_figure.exists():
    display(Image(filename=str(observed_figure)))

## 7. Produtos e supermercados
As tabelas incluem contagens, falsos positivos e falsos negativos. Para gráficos e rankings, exige-se pelo menos 30 observações; mesmo assim, diferenças podem ser instáveis e não foram ajustadas por incerteza ou multiplicidade.

In [ ]:
for col, label in [("product", "Produto"), ("supermarket", "Supermercado")]:
    groups = group_performance(test, col, threshold)
    columns = {col: label, "n": "Exemplos", "positives": "Altas reais",
               "precision": "Precisão", "recall": "Recall", "f1": "F1",
               "false_positives": "Falsos alertas", "false_negatives": "Altas sem alerta"}
    display(groups[list(columns)].sort_values("n", ascending=False).rename(columns=columns))
    display(Image(filename=str(EXPERIMENT/"figures"/f"{col}_performance.png")))

## 8. Análise de erros
Exemplos com identificadores, mês de previsão e alvo, variação observada, probabilidade e classes. Uma previsão positiva equivocada é falso alarme; uma negativa equivocada deixa de antecipar a alta.

In [ ]:
for name in ["true_positive","true_negative","false_positive","false_negative"]:
    print(name); display(pd.read_csv(EXPERIMENT/"reports"/f"{name}_examples.csv").head(5))

## 9. Interpretação: permutação, e não coeficientes
O perfil de risco por faixa de variação, calculado no **treino**, mostra o mecanismo que o modelo explora: reversão à média após quedas, com o mínimo de risco no regime de preço inalterado e uma elevação modesta em 2 a 5%. É esse ponto anguloso em zero que a fronteira linear não representa.

A importância por permutação mede a queda da *average precision* ao embaralhar cada atributo de entrada. Ela é comparável entre famílias, atua antes do one-hot e não depende da forma funcional do modelo.

Coeficientes não servem aqui. O diagnóstico de colinearidade mostra que `rolling_mean_3m` é, por construção, a média de `price_current`, `price_lag_1` e `price_lag_2` — uma combinação linear **exata**, com VIF infinito. A repartição de peso entre variáveis perfeitamente colineares é matematicamente indeterminada. A predição não é afetada; apenas a leitura atributo a atributo é.

Importância por permutação continua sendo associação, não causa: um atributo redundante pode mostrar importância baixa só porque outro carrega a mesma informação.

In [ ]:
display(Image(filename=str(EXPERIMENT/"figures"/"risk_by_variation.png")))
display(exp["risk_profile"].style.format({"risk":"{:.3f}"}).hide(axis="index"))
display(Image(filename=str(EXPERIMENT/"figures"/"permutation_importance.png")))
display(exp["collinearity"].round(3))
if (EXPERIMENT/"figures"/"coefficients.png").exists():
    print("A família vencedora é linear; os coeficientes são exibidos apenas com a ressalva de colinearidade acima.")
    display(Image(filename=str(EXPERIMENT/"figures"/"coefficients.png")))

## 10. Limitações e resposta à questão
A amostra condiciona-se à continuidade de três meses de histórico e disponibilidade do alvo, podendo selecionar séries mais estáveis. Marcas ausentes podem misturar itens; unidades e descrições têm ambiguidades. A coleta posterior a outubro/2025 não tem data individual registrada, o que impede verificar sua disponibilidade efetiva em 10 dos 12 meses-alvo do teste. Inflação, custos, promoções e mudanças de coleta não estão explicitamente representados. Comparações por grupo são descritivas e não controlam multiplicidade; resultados não garantem movimentos futuros.

In [ ]:
m=exp["metrics"]["test"]; cm=m["confusion_matrix"]
zero=exp["metrics"]["reference_test_always_zero"]; regra=exp["metrics"]["reference_test_previous_increase"]
ganho=exp["metrics"]["test_confidence_intervals"]["intervals"]["f1_gain_over_previous_increase"]
f1=exp["metrics"]["test_confidence_intervals"]["intervals"]["f1"]
display(Markdown(
    f"Família selecionada: **{exp['config']['selection']['algorithm']}**, limiar {exp['config']['selection']['threshold']:.2f}.\n\n"
    f"No teste, o recall foi **{m['recall']:.1%}**, precisão **{m['precision']:.1%}**, F1 **{m['f1']:.3f}** (IC 95% [{f1['low']:.3f}, {f1['high']:.3f}]) "
    f"e precisão média **{m['average_precision']:.3f}** (prevalência {m['positives']/m['n']:.1%}). "
    f"Houve **{cm[0][1]} falsos positivos** e **{cm[1][0]} falsos negativos**: {cm[0][1]/(cm[0][1]+cm[1][1]):.1%} dos alertas foram falsos.\n\n"
    f"A referência sempre zero teve acurácia {zero['accuracy']:.1%} com recall zero; repetir a alta anterior teve F1 {regra['f1']:.3f}. "
    f"O ganho de F1 sobre essa regra tem IC 95% de **[{ganho['low']:+.3f}, {ganho['high']:+.3f}]**, que exclui zero.\n\n"
    f"Conclusão: há sinal preditivo real nesta amostra e período, com ganho estatisticamente significativo sobre as referências, "
    f"mas com muitos alarmes falsos. Isso não garante generalização nem utilidade prática — o custo de cada tipo de erro não está nesta base."))